# 05 — Audit SCP3078 integrated MCAO atlas access and source independence

SCP3078 is a published **106,905-cell integration** of mouse tMCAO experiments (sham, day 1/3/7/14).

This notebook **does not assume it is an independent validation set**. It must recover the source datasets and study membership first, to prevent counting an input dataset a second time.

The public study metadata and export links are probed before attempting large downloads. If access is unavailable, the run records the exact block rather than reporting invented results.

Source: https://singlecell.broadinstitute.org/single_cell/study/SCP3078/mcaoatlas-allcells


In [ ]:
import json,requests,os
from pathlib import Path
ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_pet_atlas" if Path("/content").exists() else "imacs_stroke_pet_atlas"))
OUT=ROOT/"results";OUT.mkdir(parents=True,exist_ok=True)
base="https://singlecell.broadinstitute.org/single_cell"
urls=[
 f"{base}/api/v1/site/studies/SCP3078",
 f"{base}/api/v1/studies/SCP3078",
 f"{base}/api/v1/site/studies/SCP3078/files",
]
attempts=[]
results=[]
for url in urls:
 try:
    response=requests.get(url,timeout=45,headers={"Accept":"application/json"})
    text=response.text[:300]
    attempts.append({"url":url,"status_code":response.status_code,
       "content_type":response.headers.get("Content-Type",""),"preview":text})
    if response.ok and "json" in response.headers.get("Content-Type",""):
        results.append({"url":url,"metadata":response.json()})
 except Exception as exc:
    attempts.append({"url":url,"error":type(exc).__name__+":"+str(exc)[:200]})
(OUT/"05_atlas_public_API_probe.json").write_text(json.dumps(attempts,indent=2))
for x in attempts:print(x)
print("Valid JSON responses:",len(results))

In [ ]:
def collect_names(obj,prefix=""):
   hits=[]
   if isinstance(obj,dict):
       for k,v in obj.items():
           if isinstance(v,(str,int,float)) and any(t in k.lower() for t in ("file","name","url","accession","study")):
               hits.append((prefix+"."+k,str(v)[:300]))
           elif isinstance(v,(dict,list)):
               hits.extend(collect_names(v,prefix+"."+k))
   elif isinstance(obj,list):
       for i,v in enumerate(obj[:500]):hits.extend(collect_names(v,prefix+f"[{i}]"))
   return hits

rows=[]
for r in results:
 for k,v in collect_names(r["metadata"]):
    rows.append({"api_url":r["url"],"json_field":k,"value":v})
import pandas as pd
manifest=pd.DataFrame(rows,columns=["api_url","json_field","value"])
manifest.to_csv(OUT/"05_atlas_public_file_and_source_manifest.csv",index=False)
display(manifest.head(150))

report={"accession":"SCP3078","metadata_endpoints_checked":len(attempts),
 "json_responses":len(results),
 "individual_source_dataset_independence":"UNVERIFIED—do not call integrated atlas independent replication",
 "count_level_analysis":"NOT EXECUTED UNTIL H5AD DOWNLOAD VERIFIED AND SOURCE STUDIES RESOLVED"}
(OUT/"05_atlas_access_audit.json").write_text(json.dumps(report,indent=2))


## Gate for full atlas analysis

Proceed to a gene-by-study-by-time matrix only when:
1. Public H5AD counts/normalized expression file URL verified.
2. Source-study IDs, animal IDs (if available) and time labels obtained.
3. Overlaps with GSE174574 / GSE227651 identified and treated as **non-independent**.
4. Analysis respects study unit and does not produce cell-level significance.

The integrated embedding itself is **not** an independent experiment.